In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Dropout, Bidirectional, TimeDistributed
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.sequence import pad_sequences
import warnings

import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

I0000 00:00:1785049024.284925    6947 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785049024.287503    6947 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785049024.694955    6947 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785049026.316316    6947 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [3]:
df = pd.read_csv('/home/suraj/Downloads/dataset/text data/train.txt',sep=';',header=None, names=['text','emotion'])

In [4]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [5]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [6]:
unique_emotion = df['emotion'].unique()
emotion_number = {}
for i, emotion in enumerate(unique_emotion):
    emotion_number[emotion] = i

df['emotion'] = df['emotion'].map(emotion_number)



In [7]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [8]:
df['text'] = df['text'].apply(lambda x: x.lower())

In [9]:
df['text'] = df['text'].str.replace(r'[^\w\s]', '', regex=True)

In [10]:
df['text'] = df['text'].str.replace(r'\d+', '', regex=True).str.strip()

In [11]:
df['text'] = df['text'].str.replace(r'http\S+|www\.\S+', '', regex=True).str.strip()

In [12]:
df['text'] = df['text'].str.replace(r'<.*?>', '', regex=True)

In [13]:
df['text'] = df['text'].str.replace(
    r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\u2600-\u26FF\u2700-\u27BF]+',
    '',
    regex=True
)
df['text'] = df['text'].str.replace(r'[^A-Za-z0-9\s]', '', regex=True).str.strip()

In [14]:
import nltk
from nltk.corpus import stopwords


# Download stopwords if not already present
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# Tokenize and remove stopwords
df['text'] = df['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

[nltk_data] Downloading package stopwords to /home/suraj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [16]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [17]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

Training set size: 12800
Testing set size: 3200


In [21]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(max_features=5000)

X_train_count = count_vectorizer.fit_transform(X_train)
X_test_count = count_vectorizer.transform(X_test)

print(f"Training data shape: {X_train_count.shape}")
print(f"Testing data shape: {X_test_count.shape}")

nb_count_model = MultinomialNB()
nb_count_model.fit(X_train_count, y_train)

y_pred_count = nb_count_model.predict(X_test_count)

print("Accuracy (CountVectorizer):", accuracy_score(y_test, y_pred_count))
print(classification_report(y_test, y_pred_count, target_names=unique_emotion))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_count))

Training data shape: (12800, 5000)
Testing data shape: (3200, 5000)
Accuracy (CountVectorizer): 0.8384375
              precision    recall  f1-score   support

     sadness       0.85      0.93      0.89       946
       anger       0.88      0.81      0.84       427
        love       0.85      0.54      0.66       296
    surprise       0.85      0.31      0.45       113
        fear       0.82      0.77      0.79       397
         joy       0.82      0.94      0.87      1021

    accuracy                           0.84      3200
   macro avg       0.85      0.72      0.75      3200
weighted avg       0.84      0.84      0.83      3200

Confusion Matrix:
 [[882  17   6   0  12  29]
 [ 41 345   0   1  20  20]
 [ 18   6 161   1   6 104]
 [ 28   0   4  35  22  24]
 [ 45  12   1   2 305  32]
 [ 27  13  17   2   7 955]]


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a TfidfVectorizer instance
vectorizer = TfidfVectorizer(max_features=5000)

# Fit and transform the training data
X_train_vectorized = vectorizer.fit_transform(X_train)

# Transform the test data using the same vectorizer
X_test_vectorized = vectorizer.transform(X_test)

print(f"Training data shape: {X_train_vectorized.shape}")
print(f"Testing data shape: {X_test_vectorized.shape}")

Training data shape: (12800, 5000)
Testing data shape: (3200, 5000)


In [20]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_vectorized, y_train)

y_pred = nb_model.predict(X_test_vectorized)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=unique_emotion))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.7190625
              precision    recall  f1-score   support

     sadness       0.72      0.94      0.82       946
       anger       0.91      0.49      0.64       427
        love       1.00      0.11      0.21       296
    surprise       1.00      0.02      0.03       113
        fear       0.91      0.41      0.56       397
         joy       0.66      0.98      0.79      1021

    accuracy                           0.72      3200
   macro avg       0.87      0.49      0.51      3200
weighted avg       0.79      0.72      0.67      3200

Confusion Matrix:
 [[ 890    4    0    0    2   50]
 [ 119  209    0    0    5   94]
 [  50    3   34    0    2  207]
 [  43    0    0    2    7   61]
 [ 111   13    0    0  161  112]
 [  15    1    0    0    0 1005]]


In [ ]:
from sklearn.linear_model import LogisticRegression

# Create and train Logistic Regression model with TF-IDF vectorized data
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_vectorized, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test_vectorized)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, target_names=unique_emotion))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

Accuracy: 0.8690625
              precision    recall  f1-score   support

     sadness       0.90      0.94      0.92       946
       anger       0.90      0.82      0.86       427
        love       0.90      0.64      0.75       296
    surprise       0.89      0.49      0.63       113
        fear       0.85      0.78      0.81       397
         joy       0.83      0.96      0.89      1021

    accuracy                           0.87      3200
   macro avg       0.88      0.77      0.81      3200
weighted avg       0.87      0.87      0.86      3200

Confusion Matrix:
 [[891  15   2   0  10  28]
 [ 33 352   0   0  13  29]
 [  9   4 189   0   4  90]
 [ 15   0   1  55  24  18]
 [ 26  17   1   7 309  37]
 [ 13   4  16   0   3 985]]
